In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
# El Manager instala el driver compatible con la versi n de Chrome instalada
    driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
     )
    
    driver.get("https://www.google.com")
    print(f" CONECTADO ! T t u l o de la p g i n a: {driver.title}")
    driver.quit()
    
except Exception as e:
    print(f" Error: {e}")

 CONECTADO ! T t u l o de la p g i n a: Google


In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

print("=== SCRAPER DINÁMICO CON PAGINACIÓN ===\n")

# Configuración del navegador
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# Configuración del sitio
URL_INICIAL = "https://books.toscrape.com/"
SELECTOR_PRODUCTO = "article.product_pod"
SELECTOR_TITULO = "h3 a"
SELECTOR_PRECIO = "p.price_color"
SELECTOR_SIGUIENTE = "li.next a"

datos_totales = []
paginas_objetivo = 3

try:
    print(f"Conectando a: {URL_INICIAL}")
    driver.get(URL_INICIAL)
    time.sleep(3)
    
    for pagina in range(paginas_objetivo):
        print(f"\n--- Procesando Página {pagina + 1} ---")
        
        # Esperar que carguen los productos
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, SELECTOR_PRODUCTO))
        )
        
        # Capturar productos
        productos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_PRODUCTO)
        print(f"Productos encontrados: {len(productos)}")
        
        # Extraer datos
        for producto in productos:
            try:
                titulo_element = producto.find_element(By.CSS_SELECTOR, SELECTOR_TITULO)
                titulo = titulo_element.get_attribute("title")
                if not titulo:
                    titulo = titulo_element.text
                
                precio = producto.find_element(By.CSS_SELECTOR, SELECTOR_PRECIO).text.strip()
                
                datos_totales.append({
                    "titulo": titulo,
                    "precio": precio,
                    "pagina": pagina + 1,
                    "grupo": "E-commerce",
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
                })
            except Exception:
                continue
        
        # Navegar a la siguiente página
        try:
            boton_siguiente = driver.find_element(By.CSS_SELECTOR, SELECTOR_SIGUIENTE)
            boton_siguiente.click()
            time.sleep(3)
        except:
            print("No hay más páginas")
            break
    
    # Resultados
    print(f"\n{'='*50}")
    print(f"Total de productos capturados: {len(datos_totales)}")
    
    if datos_totales:
        df = pd.DataFrame(datos_totales)
        df.to_csv("libros_books_toscrape.csv", index=False)
        print(f"Archivo CSV guardado: libros_books_toscrape.csv")
        print("\nPrimeros 5 productos:")
        for i, item in enumerate(datos_totales[:5]):
            print(f"  {i+1}. {item['titulo']} - {item['precio']}")

except Exception as e:
    print(f"Error durante la ejecución: {e}")

finally:
    driver.quit()
    print("\nNavegador cerrado.")

=== SCRAPER DINÁMICO CON PAGINACIÓN ===

Conectando a: https://books.toscrape.com/

--- Procesando Página 1 ---
Productos encontrados: 20

--- Procesando Página 2 ---
Productos encontrados: 20

--- Procesando Página 3 ---
Productos encontrados: 20

Total de productos capturados: 60
Archivo CSV guardado: libros_books_toscrape.csv

Primeros 5 productos:
  1. A Light in the Attic - £51.77
  2. Tipping the Velvet - £53.74
  3. Soumission - £50.10
  4. Sharp Objects - £47.82
  5. Sapiens: A Brief History of Humankind - £54.23

Navegador cerrado.
